# LongFlow — Gate Night 3 (teacher-only: pacing remedies + 7B)

Runtime: **L4 GPU** (the 7B cell may need A100 — see cell 6). Teacher-only, no
training. ~2–2.5 h, ~$3–5. Follows N8 + Gate Night 2 (chunking NOT a cure).
Pre-registered criteria: `experiments/p1_flow_head/NOTES.md` (Gate Night 3
entry). Baselines reused from GN2 (same stack, seeded): natural ≈165–169 wpm,
monolithic-full ≈231–233 wpm.

| cell | arm | decides |
|---|---|---|
| 4 | **T1 turn-split**: same 3229 words, ONE call, ~60-word `Speaker 1:` turns, both prompts | does Microsoft's actual remedy work? If yes → pace is **per-turn scoped**, N8 localizes to single-speaker narration |
| 6 | **T2 stretched prompt**: voice prompt time-stretched 0.8×/0.9×, monolithic full script | does prompt-pace transfer hold on long scripts, and what does it cost in identity? |
| 8 | **T3 7B**: 119w + 1500w monolithic on VibeVoice-7B (mirror weights) | does the defect exist at 7B scale? |

Colab generates; Mac scores (`score_gate_night3.py`). Download
`gate_night3_bundle.zip` at the end.


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

def load_vv(model_id):
    m = VibeVoiceForConditionalGenerationInference.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, device_map="cuda"
    )
    m.eval()
    return m, VibeVoiceProcessor.from_pretrained(model_id)

MODEL_ID = "microsoft/VibeVoice-1.5B"
model, processor = load_vv(MODEL_ID)

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, sys, time
import numpy as np
import soundfile as sf

TRAIN_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
OUT = "/content/gate_night3"
os.makedirs(OUT, exist_ok=True)

def gen_raw(script_text, prompt_wav, max_new, seed=0):
    # script_text is ALREADY speaker-formatted ("Speaker 1: ...\n" lines)
    torch.manual_seed(seed)
    inputs = processor(text=[script_text], voice_samples=[[prompt_wav]],
                       return_tensors="pt", padding=True)
    inputs = {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, tokenizer=processor.tokenizer,
                             cfg_scale=1.3, max_new_tokens=max_new)
    wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    return wav, time.time() - t0

def run(tag, script_text, prompt_wav, max_new):
    wav, wall = gen_raw(script_text, prompt_wav, max_new)
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    row = {"tag": tag, "prompt": os.path.basename(prompt_wav),
           "audio_s": round(len(wav)/24000, 1), "wall_s": round(wall)}
    report["runs"].append(row)
    print(row, flush=True)

report = {"runs": []}
print("READY")


In [ ]:
# Same sentence pool + order as GN1 cell 7 / GN2 (comparable baselines).
sents = []
for f in sorted(glob.glob(f"{TRAIN_CACHE_DIR}/*.pt"))[-300:]:
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
full_words = sum(len(s.split()) for s in pool)

def prefix_script(target_words):
    words, take = 0, []
    for s in pool:
        take.append(s); words += len(s.split())
        if words >= target_words: break
    return " ".join(take)

# monolithic single-block format (as GN2)
def mono(text):
    return f"Speaker 1: {text}\n"

# turn-split format: SAME text, ~60-word same-speaker turns, ONE call
def turnsplit(target=60):
    turns, cur, w = [], [], 0
    for s in pool:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append("Speaker 1: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append("Speaker 1: " + " ".join(cur))
    return "\n".join(turns) + "\n", len(turns)

TURNSCRIPT, n_turns = turnsplit()
print(f"full={full_words} words; turn-split into {n_turns} turns")
report["full_words"] = full_words
report["n_turns"] = n_turns
prompts = sorted(glob.glob(f"{EVAL_CACHE_DIR}/*_prompt.wav"))[:2]
report["prompts"] = [os.path.basename(p) for p in prompts]


## 3. T1 — turn-split, one call (Microsoft's actual remedy, finally measured)

GN2 tested monolithic blocks and separate calls; the official docs tip — many
short same-speaker turns in a single call — is a third condition, untested by
anyone. Josh's conference app runs this shape daily with no audible rushing.
~40–70 min (longer if it works, since slower speech = more frames).


In [ ]:
for pi, prompt in enumerate(prompts):
    run(f"t1_turnsplit_p{pi}", TURNSCRIPT, prompt, 12000)


## 5. T2 — stretched-prompt transfer (the ComfyUI v1.5.0 trick, measured on long scripts)

Time-stretch the voice prompt (pitch-preserved) to 0.8× and 0.9× speed, then
render the monolithic full script. If output wpm tracks the prompt's pace, the
mimicry channel works at long context and composes with other fixes; the ECAPA
cost of stretching is measured Mac-side. ~30–40 min.


In [ ]:
import librosa
y, sr = sf.read(prompts[0], dtype="float32")
for rate in (0.8, 0.9):
    slowed = librosa.effects.time_stretch(y, rate=rate)
    path = f"/content/prompt_stretch{int(rate*100)}.wav"
    sf.write(path, slowed, sr)
    run(f"t2_stretch{int(rate*100)}", mono(" ".join(pool)), path, 12000)


## 6. T3 — 7B replication (119w + 1500w monolithic)

Mirror weights (`vibevoice/VibeVoice-7B` — original pulled). bf16 7B is tight
on L4: if this cell OOMs, switch runtime to **A100**, rerun cold start, then
run ONLY cells 2 + this one (report/bundle merge fine — earlier wavs are on
disk only if same session; on a fresh session just re-download both bundles).
~30–50 min on L4 if it fits.


In [ ]:
del model
torch.cuda.empty_cache()
MODEL_ID = "vibevoice/VibeVoice-7B"
model, processor = load_vv(MODEL_ID)
report["t3_model"] = MODEL_ID

run("t3_7b_119", mono(prefix_script(100)), prompts[0], 800)
run("t3_7b_1500", mono(prefix_script(1500)), prompts[0], 6000)


## 7. Bundle

Grab `gate_night3_bundle.zip`. Mac side:
`unzip -o ~/Downloads/gate_night3_bundle.zip -d experiments/p1_flow_head/audio/gate_night3`
then `.venv/bin/python experiments/p1_flow_head/score_gate_night3.py`.


In [ ]:
import zipfile
json.dump(report, open(f"{OUT}/gate_night3_report.json", "w"), indent=2)
with zipfile.ZipFile("/content/gate_night3_bundle.zip", "w") as z:
    for f in glob.glob(f"{OUT}/*"):
        z.write(f, os.path.basename(f))
print("download /content/gate_night3_bundle.zip")
